In [ ]:
# ===============================================================================================
# CELL 1: Wikidata Downward Crawler & Semantic Path Builder
#
# This script uses Breadth-First Search (BFS) to crawl specific domain seeds in Wikidata.
# It implements dynamic caching to prevent redundant API calls for overlapping polyhierarchies,
# and uses strict lexical filtering to amputate geographic and administrative "slices" 
# (e.g., 'Religion in Europe', 'Buddhism in Japan') protecting the pure semantic taxonomy.
# ===============================================================================================

import os
import sys
import requests
import pandas as pd
import time
from collections import deque
from dotenv import load_dotenv

# --- 1. Environment & Directory Setup ---
notebook_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in locals() else os.getcwd()
config_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "config"))
raw_data_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "data", "raw"))
interim_data_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "data", "interim"))
error_log_dir = os.path.abspath(os.path.join(interim_data_dir, "logs", "errors"))

os.makedirs(raw_data_dir, exist_ok=True)
os.makedirs(error_log_dir, exist_ok=True)

load_dotenv(os.path.join(config_dir, ".env")) 
contact_email = os.getenv("CONTACT_EMAIL", "anonymous@example.com")
sys.path.append(config_dir)

from ingest_schema_manager import get_registry_info, finalize_row, COLUMN_ORDER

SOURCE_PREFIX = "WIKIDATA"
registry_data = get_registry_info(prefix=SOURCE_PREFIX, config_dir=config_dir)
SOURCE_NAME = registry_data['Source_Name']
BASE_URI = registry_data['Base_URI']

raw_ingest_file = os.path.join(interim_data_dir, f"wikidata/raw_{SOURCE_PREFIX.lower()}_initial.csv")
discovery_file = os.path.join(interim_data_dir, f"wikidata/lateral_discovery_candidates_{SOURCE_PREFIX.lower()}.csv")
geo_drop_file = os.path.join(interim_data_dir, f"wikidata/geographic_drops_{SOURCE_PREFIX.lower()}.csv")
hard_failure_file = os.path.join(error_log_dir, f"{SOURCE_PREFIX.lower()}_hard_failures.csv")

HEADERS = {
    'User-Agent': f'ReligiousMappingProject/1.0 (mailto:{contact_email})',
    'Accept': 'application/json'
}

# --- 2. Scope & Guardrails ---
# Core upper-ontology hubs representing identities, beliefs, practices, organizations, and objects
TARGET_SEEDS = [
    'Q9174',       # religion
    'Q28855038',   # supernatural being
    'Q2110808',    # religious behaviour
    'Q3111753',    # religious person
    'Q17573152',   # believer
    'Q24398318',   # religious building
    'Q115921911',  # religious or ceremonial object
    'Q105889895',  # religious site
    'Q1530022',    # religious organization
    'Q71966963',   # religion or world view
    'Q179461'      # religious text
]
MAX_DEPTH = 7
CORE_ROOTS = set(TARGET_SEEDS)

# Halt crawling instantly if we hit these known massive geographic noise hubs
CRAWL_BLOCKLIST = {
    'Q13198592',  # religion on the Earth
}

# Forbid the upward BFS path builder from routing breadcrumbs through these hubs
PATH_BLOCKLIST = set(CRAWL_BLOCKLIST)

# Map Wikidata Properties to external vocabulary prefixes
CROSSWALK_MAP = {
    'P486': 'MeSH', 'P244': 'LCSH', 'P1014': 'AAT', 'P5806': 'SNOMED', 
    'P343': 'LOINC', 'P1036': 'DDC', 'P1190': 'UDC', 'P3348': 'LCDGT', 
    'P10165': 'TGM', 'P12107': 'ELSST', 'P3916': 'UNESCO', 'P2163': 'FAST', 
    'P227': 'GND', 'P3921': 'EUROVOC', 'P10283': 'OpenAlex'
}

# --- 3. Global State & Caching ---
# Caches ingested nodes to allow for zero-API "pass-throughs" when a child node 
# is reached via multiple redundant parent pathways (polyhierarchy).
ingested_dict = {}
if os.path.exists(raw_ingest_file):
    existing_df = pd.read_csv(raw_ingest_file, dtype={'Concept_ID': str})
    if list(existing_df.columns) != COLUMN_ORDER:
        print(f"[!] Schema mismatch. Deleting {os.path.basename(raw_ingest_file)}...")
        os.remove(raw_ingest_file)
    else:
        valid_rows = existing_df.dropna(subset=['Concept_ID', 'Primary_Label'])
        ingested_dict = dict(zip(valid_rows['Concept_ID'], valid_rows['Primary_Label']))

global_do_not_crawl = set(ingested_dict.keys())

discovery_set = set()
if os.path.exists(discovery_file):
    discovery_df = pd.read_csv(discovery_file, dtype={'Concept_ID': str})
    if 'Concept_ID' in discovery_df.columns:
        discovery_set = set(discovery_df['Concept_ID'].dropna().tolist())

path_cache = {}
api_cache = {}
ancestry_cache = {}
retry_queue = []

# --- 4. API Wrappers ---
def query_sparql(query_string):
    url = "https://query.wikidata.org/sparql"
    for attempt in range(1, 4):
        try:
            res = requests.get(url, headers=HEADERS, params={'query': query_string, 'format': 'json'}, timeout=30)
            if res.status_code == 200:
                return res.json()
            elif res.status_code == 429:
                print(f"\n  [!] Rate limited by SPARQL. Backing off...")
        except requests.exceptions.RequestException:
            pass
        time.sleep(2 ** attempt)
    return None

def fetch_entity_metadata(qid):
    if qid in api_cache:
        return api_cache[qid]
    
    url = "https://www.wikidata.org/w/api.php"
    params = {'action': 'wbgetentities', 'ids': qid, 'format': 'json', 'props': 'labels|descriptions|aliases|claims'}
    
    for attempt in range(1, 4):
        try:
            res = requests.get(url, headers=HEADERS, params=params, timeout=30)
            if res.status_code == 200:
                data = res.json().get('entities', {}).get(qid)
                if data:
                    api_cache[qid] = data
                    return data
        except requests.exceptions.RequestException:
            pass
        time.sleep(2 ** attempt)
    return None

def get_children(qid):
    query = f"SELECT ?child WHERE {{ ?child wdt:P279 wd:{qid}. }}"
    data = query_sparql(query)
    if not data: return []
    
    children = []
    for item in data.get('results', {}).get('bindings', []):
        child_qid = item.get('child', {}).get('value', '').split('/')[-1]
        if child_qid.startswith('Q'): children.append(child_qid)
    return children

# --- 5. BFS & Absolute Ancestry Resolution ---
def get_best_label(metadata, default_qid):
    labels = metadata.get('labels', {})
    if 'en' in labels: return labels['en'].get('value', default_qid)
    elif labels: return labels[list(labels.keys())[0]].get('value', default_qid)
    return default_qid

def is_geographic_node(label, metadata):
    """
    Text-based filter scanning suffixes and all intrinsic parents.
    Solves the polyhierarchy backdoor issue by checking ALL parents of a node, 
    not just the one the crawler used to arrive there.
    """
    lower_label = label.lower()
    
    # 1. Dynamic Trap for Geographic Hubs
    toxic_suffixes = [
        " on the earth", " by country", " by region", 
        " by continent", " by territorial entity", " by city"
    ]
    for suffix in toxic_suffixes:
        if lower_label.endswith(suffix):
            return True
            
    # 2. Omnidirectional Lexical Trap (Catches structural slices like "[Parent] in [Geography]")
    parents = metadata.get('claims', {}).get('P279', [])
    for p in parents:
        p_qid = p.get('mainsnak', {}).get('datavalue', {}).get('value', {}).get('id')
        if p_qid:
            p_label = ingested_dict.get(p_qid, "").lower()
            if p_label and lower_label.startswith(f"{p_label} in "):
                return True
                
    return False

def get_absolute_ancestry(qid, depth=0):
    if qid in ancestry_cache: return ancestry_cache[qid]
    if depth > 10: return ""
        
    metadata = fetch_entity_metadata(qid)
    if not metadata: return ""
    
    label = get_best_label(metadata, qid)
    parents = metadata.get('claims', {}).get('P279', [])
    
    if parents:
        for p in parents:
            p_qid = p.get('mainsnak', {}).get('datavalue', {}).get('value', {}).get('id')
            if p_qid and p_qid not in PATH_BLOCKLIST:
                parent_path = get_absolute_ancestry(p_qid, depth + 1)
                full_path = f"{parent_path} > {label}" if parent_path else label
                ancestry_cache[qid] = full_path
                return full_path
            
    ancestry_cache[qid] = label
    return label

def get_best_path(start_qid):
    if start_qid in path_cache: return path_cache[start_qid]
        
    queue = deque([(start_qid, [], 0)])
    visited = {start_qid}
    
    while queue:
        curr_qid, path_labels, depth = queue.popleft()
        if depth > 10: continue
            
        metadata = fetch_entity_metadata(curr_qid)
        if not metadata: continue
            
        label = get_best_label(metadata, curr_qid)
        
        # Prevent upward BFS from routing breadcrumbs through known geographic hubs/slices
        if is_geographic_node(label, metadata):
            continue
        
        if curr_qid in CORE_ROOTS:
            absolute_top = get_absolute_ancestry(curr_qid)
            final_path = f"{absolute_top} > {' > '.join(path_labels[::-1])}" if path_labels else absolute_top
            path_cache[start_qid] = final_path
            return final_path
            
        claims = metadata.get('claims', {})
        parents = claims.get('P279', [])
        for p in parents:
            p_qid = p.get('mainsnak', {}).get('datavalue', {}).get('value', {}).get('id')
            if p_qid and p_qid not in visited and p_qid not in PATH_BLOCKLIST:
                visited.add(p_qid)
                new_labels = list(path_labels)
                new_labels.append(label)
                queue.append((p_qid, new_labels, depth + 1))
                
    return None

# --- 6. Main Extraction Logic ---
def process_node(qid, p_id="", current_depth=0, is_retry_pass=False):
    is_already_ingested = qid in global_do_not_crawl
    
    # Fast-pass Translucent Crawling (Saves API calls on overlaps)
    if is_already_ingested:
        if current_depth >= MAX_DEPTH: return
        cached_label = ingested_dict.get(qid, qid)
        print(f"\r[DEPTH {current_depth}] Passing Through: {cached_label[:45]:<45}", end="", flush=True)
    
    # Full Metadata Extraction
    else:
        metadata = fetch_entity_metadata(qid)
        if not metadata:
            if not is_retry_pass: retry_queue.append({'uri': qid, 'p_id': p_id, 'depth': current_depth})
            return

        primary_label = get_best_label(metadata, qid)
        
        if not primary_label or primary_label == qid:
            print(f"\n  [!] Dropping {qid}: No labels found in any language.")
            if not is_retry_pass: global_do_not_crawl.add(qid)
            return

        # --- Omnidirectional Lexical Geographic Filter ---
        if is_geographic_node(primary_label, metadata):
            print(f"\n  [!] Dropping geographic node {qid}: {primary_label}")
            
            # Fetch path specifically for the log so we have audit trails
            path_for_log = get_best_path(qid) or primary_label
            
            drop_log = {
                'Concept_ID': qid, 'Primary_Label': primary_label,
                'URI': f"http://www.wikidata.org/entity/{qid}",
                'Hierarchy_Path': path_for_log,
                'Parent_ID': p_id, 'Parent_Label': ingested_dict.get(p_id, ""),
                'Action': 'Dropped via Lexical Geo Filter'
            }
            pd.DataFrame([drop_log]).to_csv(geo_drop_file, mode='a', index=False, header=not os.path.exists(geo_drop_file), encoding='utf-8-sig')
            
            if not is_retry_pass: global_do_not_crawl.add(qid)
            return

        # --- Blocklist Check ---
        if qid in CRAWL_BLOCKLIST:
            print(f"\n  [!] Hitting geographic hub {qid}. Halting branch.")
            if not is_retry_pass: global_do_not_crawl.add(qid)
            return

        # --- Strict Semantic Path Check ---
        # If BFS returns None, the node is isolated on a geographic branch with no pure doctrinal path. Drop it.
        valid_path = get_best_path(qid)
        if not valid_path:
            print(f"\n  [!] Dropping isolated node {qid}: No non-geographic path to root.")
            
            drop_log = {
                'Concept_ID': qid, 'Primary_Label': primary_label,
                'URI': f"http://www.wikidata.org/entity/{qid}",
                'Hierarchy_Path': "NO VALID PATH",
                'Parent_ID': p_id, 'Parent_Label': ingested_dict.get(p_id, ""),
                'Action': 'Dropped: Isolated by geographic hubs'
            }
            pd.DataFrame([drop_log]).to_csv(geo_drop_file, mode='a', index=False, header=not os.path.exists(geo_drop_file), encoding='utf-8-sig')
            
            if not is_retry_pass: global_do_not_crawl.add(qid)
            return
            
        description = ""
        labels = metadata.get('labels', {})
        if 'en' in labels: description = metadata.get('descriptions', {}).get('en', {}).get('value', '')
        elif labels: description = metadata.get('descriptions', {}).get(list(labels.keys())[0], {}).get('value', '')

        aliases = metadata.get('aliases', {}).get('en', [])
        synonyms = " | ".join([a.get('value') for a in aliases]) if aliases else ""
        has_translation = "yes" if len(metadata.get('labels', {}).keys()) > 1 else ""
        
        claims = metadata.get('claims', {})
        parents = [p.get('mainsnak', {}).get('datavalue', {}).get('value', {}).get('id') for p in claims.get('P279', [])]
        parent_ids_str = " | ".join(filter(None, parents)) if parents else str(p_id)
        
        crosswalks_list = []
        for prop, prefix in CROSSWALK_MAP.items():
            for ext in claims.get(prop, []):
                val = ext.get('mainsnak', {}).get('datavalue', {}).get('value')
                if val: crosswalks_list.append(f"{prefix}:{val}")

        print(f"\r[DEPTH {current_depth}] Ingesting: {primary_label[:50]:<50}", end="", flush=True)

        extracted = {
            'Source_System': SOURCE_NAME, 'Primary_Label': primary_label, 'CURIE': f"WIKIDATA:{qid}",
            'Formal_Label': "", 'Concept_Type': 'wikibase:Item', 'Hierarchy_Path': valid_path,
            'Synonyms': synonyms, 'Description': description, 'Parent_IDs': parent_ids_str,
            'Concept_ID': qid, 'URI': f"http://www.wikidata.org/entity/{qid}",
            'Has_Translation': has_translation, 'Status': 'active', 'Crosswalks': " | ".join(crosswalks_list)
        }
        
        pd.DataFrame([finalize_row(extracted)])[COLUMN_ORDER].to_csv(
            raw_ingest_file, mode='a', index=False, header=not os.path.isfile(raw_ingest_file), encoding='utf-8-sig'
        )
        global_do_not_crawl.add(qid)
        ingested_dict[qid] = primary_label  # Update cache live

    # --- Downward Recursion & Boundary Logging ---
    if current_depth < MAX_DEPTH:
        children = get_children(qid)
        for child_qid in children:
            process_node(child_qid, p_id=qid, current_depth=current_depth + 1, is_retry_pass=is_retry_pass)
            
    elif current_depth == MAX_DEPTH:
        # We hit the floor. If it's not already logged, check if there are uningested children.
        if qid not in discovery_set:
            children = get_children(qid)
            if children:
                log_data = {
                    'Concept_ID': qid,
                    'Primary_Label': primary_label,
                    'URI': f"http://www.wikidata.org/entity/{qid}",
                    'Hierarchy_Path': valid_path if not is_already_ingested else (get_best_path(qid) or primary_label),
                    'Uningested_Child_Count': len(children),
                    'Review_Status': ''  # Leave blank for manual 'yes/no' flagging
                }
                pd.DataFrame([log_data]).to_csv(
                    discovery_file, mode='a', index=False, header=not os.path.exists(discovery_file), encoding='utf-8-sig'
                )
                discovery_set.add(qid)

# --- 7. Execution ---
print(f"Starting Wikidata API Crawl (Target Depth: {MAX_DEPTH})...")

# --- Dynamic Core Root Initialization ---
print("Initializing dynamic BFS anchors...")
for seed in TARGET_SEEDS:
    CORE_ROOTS.add(seed)
    # Fetch all immediate subclasses of the seed to prevent deep crawls from stalling near the roots
    first_level_children = get_children(seed)
    if first_level_children:
        CORE_ROOTS.update(first_level_children)
        
print(f"Dynamically cached {len(CORE_ROOTS)} Core Roots to optimize pathing.\n")

for seed in TARGET_SEEDS:
    process_node(seed, current_depth=0)

if retry_queue:
    print(f"\n\nPhase 2 Cleanup. Retrying {len(retry_queue)} failed nodes...")
    current_retries = list(retry_queue)
    retry_queue.clear()
    for task in current_retries:
        process_node(task['uri'], p_id=task['p_id'], current_depth=task['depth'], is_retry_pass=True)

print(f"\n\nDone. Final data appended to {raw_ingest_file}")
print(f"Discovery candidates logged to {discovery_file}")

In [ ]:
# ===============================================================================================
# Cell 2: Smart Merge & Deduplication Script
# 
# Note that it is only necessary to run this script if you have extracted multiple seeds separately 
# as part of Cell 1, potentially introducing duplicates because of the polyhierarchical structure of Wikidata,
# and potentially creating different paths to the same concept depending on which seed it was reached from.
# 
# This script combines multiple Wikidata extraction runs and resolves polyhierarchical duplicates. 
# Instead of blindly keeping the first duplicate it finds, it scores the `Hierarchy_Path` of each 
# overlapping concept and retains the route most conceptually relevant to the domain.
# 
# Prerequisites:
# 1. Ensure you have run your extractions and saved them to the interim directory.
# 2. The script expects the following files to exist:
#    - `data/interim/wikidata/raw_wikidata_initial.csv`
#    - `data/interim/wikidata/raw_wikidata_other.csv`
# 
# Outputs:
# 1. `raw_wikidata_merged.csv`: Your deduplicated, consolidated dataset.
# 2. `wikidata_duplicates_inspection.csv`: An audit log isolating all duplicates, showing exactly 
# which paths were `Retained` vs. `Cut` and their corresponding heuristic scores.
# 
# ===============================================================================================

import os
import pandas as pd

# --- 1. Path Setup ---
current_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in locals() else os.getcwd()
interim_data_dir = os.path.abspath(os.path.join(current_dir, "..", "..", "data", "interim"))

os.makedirs(interim_data_dir, exist_ok=True)

file_original = os.path.join(interim_data_dir, "wikidata/raw_wikidata_initial.csv")
file_other = os.path.join(interim_data_dir, "wikidata/raw_wikidata_other.csv")
output_file = os.path.join(interim_data_dir, "wikidata/raw_wikidata_merged.csv")
duplicates_file = os.path.join(interim_data_dir, "wikidata/wikidata_duplicates_inspection.csv")

print("--- Wikidata Smart Merge & Deduplication ---")

# --- 2. Load Data ---
df_original = pd.read_csv(file_original, dtype={'Concept_ID': str}) if os.path.exists(file_original) else pd.DataFrame()
df_other = pd.read_csv(file_other, dtype={'Concept_ID': str}) if os.path.exists(file_other) else pd.DataFrame()

if df_original.empty or df_other.empty:
    print("[!] Merge aborted: One or both source files are missing or empty.")
    exit()

combined_df = pd.concat([df_original, df_other], ignore_index=True)
initial_count = len(combined_df)

# --- 3. The Path Scoring Heuristic ---
def score_path(path):
    """
    Scores a hierarchy path to determine the 'best' route for the domain.
    Because a node like "Quranism" has multiple legitimate Wikidata parents (Islam vs. Textualism),
    this forces the deduplicator to keep the path most aligned with our domain ontology.
    """
    if pd.isna(path): return -100
    
    score = 0
    path_lower = str(path).lower()
    
    # 1. Heavily penalize broken or geographic paths that slipped through
    if " in " in path_lower or " by " in path_lower:
        score -= 50
        
    # 2. Reward structurally complete paths (anchored to Wikidata's absolute upper-ontology root)
    if path_lower.startswith("entity >") or path_lower.startswith("entity"):
        score += 20
        
    # 3. Reward domain-native taxonomy branches (bonus points for every relevant trunk)
    core_terms = [
        'religion', 'belief system', 'abrahamic', 'islam', 'christian', 'monotheistic',
        'buddh', 'hindu', 'mythology', 'theology', 'deity', 'spiritual', 'world view'
    ]
    for term in core_terms:
        if term in path_lower:
            score += 10
            
    # 4. Tie-breaker: Shorter paths are slightly preferred IF they have the same keywords
    # This penalizes excessive rambling/meandering through Wikidata's minor subclasses
    path_length = len(path_lower.split(" > "))
    score -= path_length 
    
    return score

# --- 4. Apply Scoring and Tagging ---
# Calculate the score for every row
combined_df['Path_Score'] = combined_df['Hierarchy_Path'].apply(score_path)

# Sort by Concept_ID and then by Path_Score (descending)
# This puts the "winning" path for each concept at the very top of its group
combined_df = combined_df.sort_values(by=['Concept_ID', 'Path_Score'], ascending=[True, False])

# Initialize everything as 'Cut'
combined_df['Resolution'] = 'Cut'

# Find the first occurrence of each Concept_ID and mark it 'Retained'
retained_mask = ~combined_df.duplicated(subset=['Concept_ID'], keep='first')
combined_df.loc[retained_mask, 'Resolution'] = 'Retained'

# --- 5. Isolate Duplicates for Inspection ---
# Identify rows where the Concept_ID appeared more than once originally
duplicate_groups_mask = combined_df.duplicated(subset=['Concept_ID'], keep=False)
inspection_df = combined_df[duplicate_groups_mask].copy()

if not inspection_df.empty:
    # Reorder columns to put the audit info front and center
    front_cols = ['Concept_ID', 'Resolution', 'Path_Score', 'Hierarchy_Path']
    remaining_cols = [c for c in inspection_df.columns if c not in front_cols]
    inspection_df = inspection_df[front_cols + remaining_cols]
    
    inspection_df.to_csv(duplicates_file, index=False, encoding='utf-8-sig')
    print(f"\n[*] Isolated {len(inspection_df)} overlapping rows for review.")
    print(f"    Saved audit log to: {os.path.basename(duplicates_file)}")

# --- 6. Extract Final Dataset ---
# Keep only the rows flagged as 'Retained'
deduped_df = combined_df[combined_df['Resolution'] == 'Retained'].copy()

# Clean up the temporary tracking columns for the final output
deduped_df = deduped_df.drop(columns=['Path_Score', 'Resolution'])

final_count = len(deduped_df)
duplicates_removed = initial_count - final_count

# --- 7. Reporting & Export ---
print("\n--- Results ---")
print(f"Total rows before deduplication: {initial_count}")
print(f"Overlapping duplicates removed:  {duplicates_removed}")
print(f"Final unique concepts to review: {final_count}")

deduped_df.to_csv(output_file, index=False, encoding='utf-8-sig')
print(f"\n[+] Success! Smart-merged dataset saved to: {os.path.basename(output_file)}")

In [ ]:
# ================================================================================================
# Cell 3: Hierarchical Path-Based Pruning Engine
# 
# This script applies Human-in-the-Loop QA decisions to amputate noisy branches (like instances or trivia) 
# from the raw dataset. It uses strict prefix-matching on the Hierarchy_Path to guarantee unambiguous, 
# deterministic topological cuts.
# 
# This script reproducibly prunes the initial output of Cell 1 (or Cell 2, if you ran it). It reads your 
# manual pruning instructions and removes out-of-scope hierarchy branches. (At time of testing, about 40%
# of the dataset was out of scope).  
# 
# Prerequisite steps:
# 1. Open your output file from Cell 2 (or Cell 1 if you didn't run Cell 2) in Excel or a CSV editor.
# 2. Add a new column to the far right named exactly: `Prune_Action`.
# 3. Sort the file by `Hierarchy_Path` to group similar branches together and make it easier to spot patterns.
# 4. Scroll through the dataset and mark specific rows to prune unwanted branches, using the following flags:
#   - `floor`:     Keep this node, Drop all its descendants.
#   - `prune`:     Drop this node, Drop all its descendants.
#   - `exception`: Keep this node, Keep all its descendants (Overrides a higher floor/prune).
#   - `keepnode`:  Keep this node, Drop all its descendants (Overrides a higher floor/prune).
# Note that not every cell needs to have a flag; you only need to mark the top-level nodes of the branches 
# that you want to prune. 
# 5. Save the marked-up file as `wikidata_pruning_instructions.csv` in the `data/interim/wikidata/` folder.
# 
# Outputs:
# 1. `data/raw/raw_wikidata_survivors.csv`: A final Wikidata ingestion dataset suitable for consolidation
# and categorization steps. 
# 2. `data/interim/wikidata/raw_wikidata_pruned.csv`: An archive of the specific concepts that were amputated during this run.
# 
# ================================================================================================


import os
import pandas as pd

# --- 1. Path Setup ---
current_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in locals() else os.getcwd()
interim_data_dir = os.path.abspath(os.path.join(current_dir, "..", "..", "data", "interim"))
raw_data_dir = os.path.abspath(os.path.join(current_dir, "..", "..", "data", "raw"))

os.makedirs(raw_data_dir, exist_ok=True)
os.makedirs(interim_data_dir, exist_ok=True)

input_file = os.path.join(interim_data_dir, "wikidata/wikidata_pruning_instructions.csv")
survivors_file = os.path.join(raw_data_dir, "raw_wikidata_survivors.csv")
pruned_file = os.path.join(interim_data_dir, "wikidata/raw_wikidata_pruned.csv")

print("--- Initializing Strict Hierarchy Path Pruning Engine ---")

# --- 2. Load Data ---
df = pd.read_csv(input_file, dtype={'Concept_ID': str})

if 'Prune_Action' in df.columns:
    df['Prune_Action'] = df['Prune_Action'].fillna('').astype(str).str.lower().str.strip()
else:
    print("[!] Error: Could not find 'Prune_Action' column in the CSV.")
    exit()

# --- 3. Build the Rule Dictionary ---
# ACTION MATRIX EXPLANATION:
# - 'floor':     Keep this node, Drop all descendants.
# - 'prune':     Drop this node, Drop all descendants.
# - 'exception': Keep this node, Keep all descendants (punches through a higher prune/floor).
# - 'keepnode':  Keep this node, Drop all descendants (punches through a higher prune/floor).
rules = {}
marked_rows = df[df['Prune_Action'].isin(['floor', 'prune', 'exception', 'keepnode'])]

for _, row in marked_rows.iterrows():
    path = str(row['Hierarchy_Path']).strip()
    action = row['Prune_Action']
    
    if path != "nan" and path:
        rules[path] = action

print(f"[*] Loaded {len(rules)} strict path-based instructions.")

# --- 4. The Evaluation Logic ---
def evaluate_row(row):
    path = str(row['Hierarchy_Path']).strip()
    
    # If a node somehow has no path, default to keeping it
    if path == "nan" or not path:
        return True 
        
    matches = []
    
    # Compare this row's path against every established rule
    for rule_path, rule_action in rules.items():
        if path == rule_path:
            # The rule targets this exact node
            matches.append((len(rule_path), rule_action, 'self'))
        elif path.startswith(rule_path + " > "):
            # This node is a descendant of the rule's target
            matches.append((len(rule_path), rule_action, 'descendant'))
            
    if not matches:
        return True # Default to KEEP if no rules apply
        
    # Sort matches by the length of the rule_path (descending)
    # The longest matching prefix is the deepest/closest ancestor to this node.
    # This ensures specific branch rules override broad trunk rules.
    matches.sort(key=lambda x: x[0], reverse=True)
    
    deepest_action = matches[0][1]
    match_type = matches[0][2]
    
    # --- Execute Action Matrix ---
    if match_type == 'self':
        return deepest_action in ['floor', 'exception', 'keepnode']
        # 'prune' returns False
        
    elif match_type == 'descendant':
        return deepest_action == 'exception'
        # 'floor', 'prune', and 'keepnode' all return False for descendants

# --- 5. Apply Rules and Clean Columns ---
print("[*] Parsing hierarchy branches...")
df['Survives'] = df.apply(evaluate_row, axis=1)

# Split into dataframes and drop the calculation and QA columns to preserve the strict 15-column schema
columns_to_drop = ['Survives', 'Prune_Action']
survivors_df = df[df['Survives'] == True].drop(columns=columns_to_drop, errors='ignore')
pruned_df = df[df['Survives'] == False].drop(columns=columns_to_drop, errors='ignore')

# --- 6. Export ---
survivors_df.to_csv(survivors_file, index=False, encoding='utf-8-sig')
pruned_df.to_csv(pruned_file, index=False, encoding='utf-8-sig')

print("\n--- Pruning Complete ---")
print(f"Original Concepts: {len(df)}")
print(f"Survivors:         {len(survivors_df)} -> saved to 'data/raw/'")
print(f"Pruned (Removed):  {len(pruned_df)} -> saved to 'data/interim/'")